# Lab 2.4 - Pruning, Conversion, and Deployment to Raspberry Pi Pico

This notebook has `# TODO` markers (TODO 1-8). Work through them in order.

Copy your `mnist_cnn.h5` from Lab 2.2 into this same folder before you start.

In [1]:
import os
import time
import gzip
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras
import tensorflow_model_optimization as tfmot
import matplotlib.pyplot as plt

tf.random.set_seed(10)
np.random.seed(10)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Step 0 - Load data and the Lab 2.2 model

This part is provided for you.

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train_n = (x_train / 255.0).astype("float32")[..., np.newaxis]
x_test_n  = (x_test  / 255.0).astype("float32")[..., np.newaxis]

model = tf_keras.models.load_model("mnist_cnn.h5")
_, baseline_acc = model.evaluate(x_test_n, y_test, verbose=0)
baseline_size_kb = os.path.getsize("mnist_cnn.h5") / 1024
print(f"Float32 baseline -- size: {baseline_size_kb:.1f} KB, accuracy: {baseline_acc:.4f}")

    8192/11490434 [..............................] - ETA: 0s

   16384/11490434 [..............................] - ETA: 1:29

   49152/11490434 [..............................] - ETA: 47s 

   81920/11490434 [..............................] - ETA: 37s

  122880/11490434 [..............................] - ETA: 29s

  172032/11490434 [..............................] - ETA: 24s

... (37 outputs left out)

10919936/11490434 [===========================>..] - ETA: 0s

11165696/11490434 [============================>.] - ETA: 0s

11490434/11490434 [==============================] - 3s 0us/step


Float32 baseline -- size: 212.1 KB, accuracy: 0.9869


## Part A - Pruning

`prune_low_magnitude` wraps every prunable layer so that, during training, the
smallest-magnitude weights in each layer are progressively zeroed out
following a schedule.

**TODO 1:** Build a `PolynomialDecay` pruning schedule: start at 0% sparsity,
ramp to a **50% final sparsity target**, over `end_step` steps (already
computed for you below - `steps_per_epoch * epochs`). Then wrap `model` with
`tfmot.sparsity.keras.prune_low_magnitude()` using that schedule, and
compile it (same loss/metric as always).

In [3]:
num_images = x_train_n.shape[0]
batch_size = 64
prune_epochs = 2
end_step = int(np.ceil(num_images / batch_size)) * prune_epochs

# TODO 1: 0% -> 50% sparsity over end_step steps
pruning_params = {
    "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0, final_sparsity=0.50, begin_step=0, end_step=end_step
    )
}
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)
pruned_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

pruned_model.summary()


Model: "sequential"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 prune_low_magnitude_conv2d  (None, 26, 26, 8)         154       


  (PruneLowMagnitude)                                            


 prune_low_magnitude_max_po  (None, 13, 13, 8)         1         


... (13 outputs left out)

Total params: 28765 (112.39 KB)


Trainable params: 14410 (56.29 KB)


Non-trainable params: 14355 (56.10 KB)


_________________________________________________________________


**TODO 2:** Fine-tune `pruned_model`. This **requires** the
`tfmot.sparsity.keras.UpdatePruningStep()` callback - it's what actually
advances the pruning schedule each step. Skip it and the model trains
normally but never actually gets pruned. Use `prune_epochs` from above.

In [4]:
# TODO 2: no validation_split, it would shorten the epochs and the schedule would stop before end_step
pruned_model.fit(
    x_train_n, y_train,
    epochs=prune_epochs,
    batch_size=batch_size,
    validation_data=(x_test_n, y_test),
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=2,
)


Epoch 1/2


938/938 - 3s - loss: 0.0464 - accuracy: 0.9858 - val_loss: 0.0483 - val_accuracy: 0.9841 - 3s/epoch - 3ms/step


Epoch 2/2


938/938 - 2s - loss: 0.0417 - accuracy: 0.9870 - val_loss: 0.0372 - val_accuracy: 0.9874 - 2s/epoch - 2ms/step


**TODO 3:** Strip the pruning wrappers with
`tfmot.sparsity.keras.strip_pruning()`, then compile and evaluate the
stripped model. **This step is mandatory** - `prune_low_magnitude` wraps each
layer in a `PruneLowMagnitude` wrapper that only exists for training-time
bookkeeping; the model won't convert to TFLite cleanly with those wrappers
still attached.

Also check the actual sparsity achieved (a loop over `trainable_weights`
checking `np.mean(w.numpy() == 0)` for kernel weights is provided below) -
confirm it's close to your 50% target.

In [5]:
# TODO 3: strip pruning wrappers, compile, evaluate, and save as pruned_model.h5
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
stripped_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
pruned_acc = stripped_model.evaluate(x_test_n, y_test, verbose=0)[1]
stripped_model.save("pruned_model.h5", include_optimizer=False)
pruned_size_kb = os.path.getsize("pruned_model.h5") / 1024

print(f"Pruned (pre-quantization) -- size: {pruned_size_kb:.1f} KB, accuracy: {pruned_acc:.4f}")

for w in stripped_model.trainable_weights:
    if "kernel" in w.name:
        sparsity = np.mean(w.numpy() == 0)
        print(f"  {w.name}: {sparsity:.1%} zeros")


Pruned (pre-quantization) -- size: 77.3 KB, accuracy: 0.9874
  conv2d/kernel:0: 50.0% zeros
  conv2d_1/kernel:0: 50.0% zeros
  dense/kernel:0: 50.0% zeros
  dense_1/kernel:0: 50.0% zeros


/opt/venv/lib/python3.12/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## Part B - Combine Pruning + Quantization

Same PTQ process as Lab 2.3, applied to the *pruned* model instead of the
original.

**TODO 4:** Configure a `TFLiteConverter` for full-integer int8 quantization
(same settings as Lab 2.3), convert `stripped_model`, and save as
`model.tflite`.

In [6]:
def representative_dataset():
    for i in range(200):
        yield [x_train_n[i:i+1]]

# TODO 4: build and configure the converter, convert, and save model.tflite
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter._experimental_disable_per_channel_quantization_for_dense_layers = True  # the Pico's TFLite Micro has one scale per Dense layer
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_pruned_quant = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_pruned_quant)

pq_size_kb = len(tflite_pruned_quant) / 1024
print(f"Pruned + quantized: {pq_size_kb:.1f} KB")


INFO:tensorflow:Assets written to: /tmp/tmp8_t_y6g1/assets


INFO:tensorflow:Assets written to: /tmp/tmp8_t_y6g1/assets


Pruned + quantized: 18.8 KB


/opt/venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1794034998.351152       8 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794034998.351324       8 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794034998.351536       8 reader.cc:83] Reading SavedModel from: /tmp/tmp8_t_y6g1
I0000 00:00:1794034998.351802       8 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794034998.351805       8 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp8_t_y6g1
I0000 00:00:1794034998.353729       8 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1794034998.354066       8 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794034998.362600       8 loader.cc:220] Running initialization op on SavedModel bundle at pa

**TODO 5:** Evaluate the pruned+quantized TFLite model (reuse your
`evaluate_tflite()` pattern from Lab 2.3 - a fresh copy is provided below to
complete). Also print the input tensor's `(scale, zero_point)` - you'll see
these same values reflected in how the Pico C++ code quantizes images.

In [7]:
def evaluate_tflite(tflite_bytes, x, y, n_latency=100):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    scale, zero_point = inp["quantization"]

    def quantize(img):
        return (img / scale + zero_point).round().astype(np.int8)

    # TODO 5a: loop over all of x/y, run inference, count correct predictions
    correct = 0
    for i in range(len(x)):
        interpreter.set_tensor(inp["index"], quantize(x[i:i+1]))
        interpreter.invoke()
        if int(np.argmax(interpreter.get_tensor(out["index"]))) == int(y[i]):
            correct += 1
    accuracy = correct / len(x)

    # TODO 5b: time n_latency invocations (same pattern as Lab 2.3)
    start = time.time()
    for i in range(n_latency):
        interpreter.set_tensor(inp["index"], quantize(x[i:i+1]))
        interpreter.invoke()
    latency_ms = (time.time() - start) * 1000.0 / n_latency

    return accuracy, latency_ms, inp["dtype"], out["dtype"], scale, zero_point

pq_acc, pq_latency_ms, pq_in_dtype, pq_out_dtype, input_scale, input_zero_point = \
    evaluate_tflite(tflite_pruned_quant, x_test_n, y_test)

print(f"Pruned + quantized -- accuracy: {pq_acc:.4f}, latency: {pq_latency_ms:.3f} ms/image, "
      f"input dtype: {pq_in_dtype}, output dtype: {pq_out_dtype}")
print(f"Input quantization -- scale: {input_scale}, zero_point: {input_zero_point}")


/opt/venv/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Pruned + quantized -- accuracy: 0.9874, latency: 0.009 ms/image, input dtype: <class 'numpy.int8'>, output dtype: <class 'numpy.int8'>
Input quantization -- scale: 0.003921568859368563, zero_point: -128


**Does pruning + quantization beat quantization alone?** Compare the raw
`.tflite` file size against Lab 2.3's PTQ-only number. TFLite stores pruned
weights as literal zero bytes in a dense array, so don't assume pruning
automatically shrinks the raw flatbuffer - check what it actually does to
size, and what happens once the file is compressed (gzip).

**A trap worth knowing about:** `prune_low_magnitude()` wraps the *same*
underlying layer objects, by reference, not copies. Fine-tuning
`pruned_model` in TODO 2 therefore also mutated `model`'s weights in place -
`model` is no longer the original float32 baseline. For a fair "unpruned"
comparison here, reload a fresh copy from `mnist_cnn.h5` rather than reusing
`model`.

In [8]:
def gzip_size_kb(path):
    with open(path, "rb") as f:
        data = f.read()
    return len(gzip.compress(data, compresslevel=9)) / 1024

# TODO 6: fresh copy of mnist_cnn.h5, because fine-tuning changed `model` in place
model_unpruned_fresh = tf_keras.models.load_model("mnist_cnn.h5")
converter = tf.lite.TFLiteConverter.from_keras_model(model_unpruned_fresh)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter._experimental_disable_per_channel_quantization_for_dense_layers = True  # the Pico's TFLite Micro has one scale per Dense layer
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_unpruned_quant = converter.convert()

with open("model_unpruned_quant.tflite", "wb") as f:
    f.write(tflite_unpruned_quant)

print(f"Unpruned + quantized: raw {len(tflite_unpruned_quant) / 1024:.1f} KB, "
      f"gzip {gzip_size_kb('model_unpruned_quant.tflite'):.1f} KB")
print(f"Pruned + quantized:   raw {pq_size_kb:.1f} KB, "
      f"gzip {gzip_size_kb('model.tflite'):.1f} KB")


INFO:tensorflow:Assets written to: /tmp/tmp9__55_l5/assets


INFO:tensorflow:Assets written to: /tmp/tmp9__55_l5/assets


Unpruned + quantized: raw 18.8 KB, gzip 14.1 KB
Pruned + quantized:   raw 18.8 KB, gzip 10.9 KB


/opt/venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1794034998.793754       8 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794034998.793769       8 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794034998.793829       8 reader.cc:83] Reading SavedModel from: /tmp/tmp9__55_l5
I0000 00:00:1794034998.794115       8 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794034998.794117       8 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp9__55_l5
I0000 00:00:1794034998.796162       8 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794034998.809273       8 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp9__55_l5
I0000 00:00:1794034998.813156       8 loader.cc:471] SavedModel load for tags { serve }; Status

## Final comparison across all 4 sessions

In [9]:
comparison = pd.DataFrame([
    {"variant": "Float32 baseline (2.2)", "size_kb": round(baseline_size_kb, 1), "accuracy": round(baseline_acc, 4)},
    {"variant": "PTQ int8 (2.3)", "size_kb": 19.8, "accuracy": 0.9870},
    {"variant": "QAT int8 (2.3)", "size_kb": 19.3, "accuracy": 0.9865},
    {"variant": "Pruned + quantized (2.4)", "size_kb": round(pq_size_kb, 1), "accuracy": round(pq_acc, 4)},
])
comparison


,variant,size_kb,accuracy
0,Float32 baseline (2.2),212.1,0.9869
1,PTQ int8 (2.3),19.8,0.9870
2,QAT int8 (2.3),19.3,0.9865
3,Pruned + quantized (2.4),18.8,0.9874


## Part C - Convert for Deployment

> **Note:** TensorFlow Lite has been rebranded **LiteRT**. The conversion API
> and file format are unchanged; only the name and some docs have moved.

`xxd -i` turns the `.tflite` flatbuffer into a C array, but the symbol names
it generates (`model_tflite`, `model_tflite_len`) don't match what
`pico_sw/src/mnist_model_data.h` declares (`mnist_model_data`,
`mnist_model_data_len`).

**TODO 7:** Run `xxd -i model.tflite` (via `subprocess`, captured as text),
then rename `model_tflite` -> `mnist_model_data` and `model_tflite_len` ->
`mnist_model_data_len` in the output. Also prepend
`#include "mnist_model_data.h"` and change `unsigned char` to
`alignas(8) const unsigned char` (TFLite Micro expects 8-byte-aligned model
data). Write the result to `mnist_model_data.cpp`.

In [10]:
import subprocess

result = subprocess.run(["xxd", "-i", "model.tflite"], capture_output=True, text=True, check=True)
xxd_output = result.stdout

# TODO 7: transform xxd_output into the final C++ source and write it to
# mnist_model_data.cpp
c_array = xxd_output.replace("unsigned char model_tflite[]", "alignas(8) const unsigned char mnist_model_data[]")
c_array = c_array.replace("unsigned int model_tflite_len", "const int mnist_model_data_len")
c_array = '#include "mnist_model_data.h"\n\n' + c_array

with open("mnist_model_data.cpp", "w") as f:
    f.write(c_array)

print(f"Wrote mnist_model_data.cpp ({len(c_array)} chars)")


Wrote mnist_model_data.cpp (119021 chars)


**TODO 8:** Sanity check - confirm `mnist_model_data_len` in your
generated file matches the actual `.tflite` file's byte size. A mismatch
here means the interpreter will read past the end of the array on-device.

In [11]:
# TODO 8: print len(tflite_pruned_quant) and confirm it matches the
# mnist_model_data_len value written into mnist_model_data.cpp above
import re

written_len = int(re.search(r"mnist_model_data_len = (\d+)", c_array).group(1))
print(f"tflite bytes: {len(tflite_pruned_quant)}, mnist_model_data_len: {written_len}")
assert written_len == len(tflite_pruned_quant)


tflite bytes: 19280, mnist_model_data_len: 19280


Copy `mnist_model_data.cpp` into `pico_sw/models/mnist_model_data.cpp`,
replacing the placeholder there (TODO 1/2 in that file).

## Deliverable checklist

- [x] Pruned + quantized `.tflite` file (`model.tflite` above)
- [x] Final comparison table across all 4 sessions (table above)
- [x] Short writeup, below

## On the Pico (Part D)

Own digits: the 10 drawn on the LCD touchscreen, sent to `main_uart_inference`.

| Model | File | Tensor arena | Time per digit | Own digits on device |
|---|---|---|---|---|
| Pruned + quantized | 18.8 KB | 8.3 KB | 31.7 ms | 7/10 |

## Which combination would I ship?

Pruned + int8. On raw flash pruning saves nothing: 18.8 KB with and without it, because the zero
weights are still stored. It only pays off after compression (gzip 14.1 KB down to 10.9 KB), and it
cost no accuracy here (98.74%). The size win on the Pico comes from quantization: 212.1 KB down to
18.8 KB.

The converter here has per-neuron Dense quantization switched off, because the Pico's TFLite Micro
applies one scale per Dense layer. With it on the file was 19.8 KB and the scores on the Pico were wrong.
